# Activation steering to flip a sentiment classifier

The goal is to find two steering vectors that, when added to layer 3 of a MiniLM encoder, flip a frozen sentiment classifier from negative to positive and the other way. The notebook is based on the provided starter code, my part is computing the steering direction as the difference between the mean layer-3 activations of positive and negative sentences. Local flip rate was 0.81, the official pre-judging script gave a final score of 0.245.

# The Whisper Vectors — starter

Read `README.md` first. This notebook is **self-contained** — everything you need is defined here. It walks through:

1. Loading the `models` pickle bundle (embedder + classifier) and wiring a forward hook at block 3.
2. Two steering tools: `embed_with_steering`, `get_layer3_activations`.
3. Loading the 50 unlabeled val sentences.
4. Baselines (zero, random) so you see what *doesn't* flip predictions.
5. A local flip-rate helper to iterate on candidate vectors.
6. `save_submission(...)` — writes the 2-row × 384-column `submission.csv`.

Finding a steering direction that actually works is yours to write.


## 1. Setup — load models + install the block-3 hook


In [79]:
import os
folder_path = r"C:\Users\asus\LeagueIV_P2"

# List everything inside
files = os.listdir(folder_path)
print("Files in folder:", files)

Files in folder: ['.ipynb_checkpoints', 'combined_data.csv', 'contestant_output.csv', 'custom_archive(2)', 'LeagueIV_Problem2.ipynb', 'pre_judging_script.py', 'test_data(1).csv', 'train_data']


In [80]:
file_path = r"C:\Users\asus\LeagueIV_P2\train_data"

with open(file_path, "rb") as f:
    bundle = pickle.load(f)
embedder = bundle["embedder"]
classifier = bundle["classifier"]

C:\Users\asus\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\asus\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [81]:
import pickle, sys, types
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
import torch

EMBED_DIM   = 384
NORM_CAP    = 1.0    # submit vectors with L2 norm strictly less than this
HOOK_BLOCK  = 3      # output of the 4th of 6 transformer blocks

# The pickled classifier class needs to be importable from both
# `classifier_wrapper` and `__main__` for pickle.load to find it.
class FrozenSentimentClassifier:
    def __init__(self, clf): self._clf = clf
    def predict(self, X):       return self._clf.predict(X)
    def predict_proba(self, X): return self._clf.predict_proba(X)

_fake = types.ModuleType('classifier_wrapper')
_fake.FrozenSentimentClassifier = FrozenSentimentClassifier
sys.modules['classifier_wrapper'] = _fake
if '__main__' in sys.modules and not hasattr(sys.modules['__main__'], 'FrozenSentimentClassifier'):
    sys.modules['__main__'].FrozenSentimentClassifier = FrozenSentimentClassifier

# Load the bundle: {'embedder': SentenceTransformer, 'classifier': FrozenSentimentClassifier}.
print('loaded models:', type(embedder).__name__, '+', type(classifier).__name__)


loaded models: SentenceTransformer + FrozenSentimentClassifier


### The steering hook

We register a single forward hook on block 3 of the MiniLM encoder. The hook:

- Optionally adds the *current* steering vector `v` to the block's output, broadcast over every token position.
- Optionally captures the block's output (used by `get_layer3_activations`).

The hook is active for the lifetime of this notebook — don't re-run this cell unless you also reload `embedder`.


In [82]:
_state = {'v': None, 'captured': None}

def _hook(module, inp, out):
    hs = out[0] if isinstance(out, tuple) else out
    if _state['captured'] is not None:
        _state['captured'].append(hs.detach().cpu().numpy().copy())
    if _state['v'] is not None:
        hs = hs + _state['v'].to(hs.device).to(hs.dtype)
    return (hs,) + out[1:] if isinstance(out, tuple) else hs

embedder[0].auto_model.encoder.layer[HOOK_BLOCK].register_forward_hook(_hook)

@contextmanager
def _steer(v):
    old = _state['v']
    _state['v'] = torch.from_numpy(np.asarray(v, dtype=np.float32))
    try: yield
    finally: _state['v'] = old

def embed_with_steering(texts, v=None):
    """Encode texts with MiniLM; if v is given, inject it at block 3."""
    enc = lambda: embedder.encode(list(texts), normalize_embeddings=False,
                                   batch_size=64, show_progress_bar=False,
                                   convert_to_numpy=True).astype(np.float32)
    if v is None:
        return enc()
    with _steer(v):
        return enc()

def get_layer3_activations(texts, pool='mean'):
    """(N, 384) activations at the injection point. pool: 'mean' or 'cls'."""
    tok = embedder.tokenizer(list(texts), padding=True, truncation=True,
                              max_length=128, return_tensors='pt')
    device = next(embedder[0].auto_model.parameters()).device
    tok = {k: v.to(device) for k, v in tok.items()}
    _state['captured'] = []
    try:
        with torch.no_grad():
            embedder[0].auto_model(**tok)
    finally:
        captured = _state['captured']; _state['captured'] = None
    hs = captured[0]
    mask = tok['attention_mask'].cpu().numpy()
    if pool == 'cls':
        return hs[:, 0, :].astype(np.float32)
    m = mask[..., None].astype(np.float32)
    return ((hs * m).sum(1) / np.clip(m.sum(1), 1e-6, None)).astype(np.float32)


## 2. Demo the two tools


In [83]:
emb = embed_with_steering(['a wonderful film'])
print('final embedding :', emb.shape, emb.dtype)
acts = get_layer3_activations(['a wonderful film'], pool='mean')
print('layer-3 activation:', acts.shape, 'norm =', float(np.linalg.norm(acts[0])))

final embedding : (1, 384) float32
layer-3 activation: (1, 384) norm = 7.6543121337890625


## 3. Load the val set

50 unlabeled sentences. By construction, the classifier predicts 25 of them negative and 25 positive — your local testbed.


In [258]:
val_df = pd.read_csv(r'C:\Users\asus\LeagueIV_P2\combined_data.csv')   # DataPointID, text
idx = 42
bomba = val_df['text'].tolist()
bomba_df = val_df.loc[:idx, :]
preds = get_layer3_activations(bomba[:idx], pool='mean')
X_val = embed_with_steering(bomba[:idx])
senses = bomba[:idx]
pred_val = classifier.predict(X_val)
print(f'val: {val_df.shape}  predictions: neg={int((pred_val==0).sum())} pos={int((pred_val==1).sum())}')

val: (53, 2)  predictions: neg=21 pos=21


## 4. Local flip-rate helper

Partition the 50 val sentences by the classifier's *current* prediction (no steering), then measure how many flip when you add your steering vectors.


In [259]:
preds

array([[ 0.42744058, -0.12019358,  0.15331748, ..., -0.02579523,
         0.44975024, -0.21649198],
       [-0.12500972, -0.24476117, -0.0741671 , ...,  0.2437913 ,
         0.25579223, -0.1146826 ],
       [ 0.06458286,  0.18368207,  0.00194117, ..., -0.17944613,
         0.46292433, -0.22853048],
       ...,
       [ 0.43001887,  0.33504897, -0.07884062, ..., -0.5594823 ,
         0.01204996,  0.6621818 ],
       [-0.5116606 ,  0.08923857,  0.41104212, ..., -0.32267344,
         0.3789147 ,  0.21071626],
       [-0.26988325, -0.00406221,  0.02922173, ..., -0.44854522,
        -0.08570663,  0.18826176]], dtype=float32)

In [260]:
X_val_tensor = torch.tensor(X_val[:42], dtype=torch.float32)
pred_val_tensor = torch.tensor(pred_val[:42], dtype=torch.int64)
X_acts = torch.tensor(preds[:42], dtype=torch.float32)

# 2. Use Boolean Masking to filter
# This replaces your 'if/else' loop entirely
good_embeddings = X_val_tensor[pred_val_tensor == 1]
bad_embeddings = X_val_tensor[pred_val_tensor == 0]

good_acts = X_acts[pred_val_tensor == 1]
bad_acts = X_acts[pred_val_tensor == 0]

# 3. Calculate means
# dim=0 means "average down the rows" to get one 384-length vector
good_class = good_embeddings.mean(dim=0)
bad_class = bad_embeddings.mean(dim=0)

good_acts = good_acts.mean(dim=0)
bad_acts = bad_acts.mean(dim=0)

In [261]:
"""
bad = torch.zeros(384, dtype=torch.float32)
good = torch.zeros(384, dtype=torch.float32)
bad_cnt = 0
good_cnt = 0

for i in range(20):
    if(pred_val[i] == np.int64(1)):
        good = good + X_val[i]
        good_cnt = good_cnt + 1
    else:
        bad = bad + X_val[i]
        bad_cnt = bad_cnt + 1

good = good / good_cnt
bad = bad / bad_cnt

print(bad_cnt)
"""

'\nbad = torch.zeros(384, dtype=torch.float32)\ngood = torch.zeros(384, dtype=torch.float32)\nbad_cnt = 0\ngood_cnt = 0\n\nfor i in range(20):\n    if(pred_val[i] == np.int64(1)):\n        good = good + X_val[i]\n        good_cnt = good_cnt + 1\n    else:\n        bad = bad + X_val[i]\n        bad_cnt = bad_cnt + 1\n\ngood = good / good_cnt\nbad = bad / bad_cnt\n\nprint(bad_cnt)\n'

In [262]:
def local_flip_rate(v_neg_to_pos, v_pos_to_neg, df=val_df):
    texts = df['text'].tolist()
    cur = classifier.predict(embed_with_steering(texts))
    pos_t = [t for t, m in zip(texts, cur == 1) if m]
    neg_t = [t for t, m in zip(texts, cur == 0) if m]
    p = (classifier.predict(embed_with_steering(pos_t, v_pos_to_neg)) == 0).mean() if pos_t else 0.0
    n = (classifier.predict(embed_with_steering(neg_t, v_neg_to_pos)) == 1).mean() if neg_t else 0.0
    return float((p+n)/2)

In [263]:
# Move from Negative cluster TO Positive cluster
v_neg_to_pos = good_acts - bad_acts

# Move from Positive cluster TO Negative cluster
v_pos_to_neg = bad_acts - good_acts

In [267]:
local_flip_rate(v_neg_to_pos, v_pos_to_neg, df=val_df)

0.8102678571428572

## 5. Baselines


In [243]:
z = np.zeros(EMBED_DIM, dtype=np.float32)
print('zero baseline  :', local_flip_rate(z, z))

rng = np.random.default_rng(0)
u = rng.standard_normal(EMBED_DIM).astype(np.float32)
v_neg_to_pos /= np.linalg.norm(v_neg_to_pos); v_neg_to_pos *= 0.999
print('random baseline:', local_flip_rate(u, -u))


zero baseline  : 0.0
random baseline: 0.046875


In [246]:
v_neg_to_pos /= np.linalg.norm(v_neg_to_pos); v_neg_to_pos *= 0.999
v_pos_to_neg /= np.linalg.norm(v_pos_to_neg); v_pos_to_neg *= 0.999
print('random baseline:', local_flip_rate(v_pos_to_neg, -v_pos_to_neg))


random baseline: 0.0


## 6. Save your submission

When you have a `(v_neg_to_pos, v_pos_to_neg)` pair you're happy with, call `save_submission(...)`. It writes `submission.csv` — 2 rows × 384 columns, no header. **Each vector's L2 norm must be strictly less than 1.0.**


In [247]:
def save_submission(v_neg_to_pos, v_pos_to_neg, path=r'C:\Users\asus\LeagueIV_P2\submission.csv'):
    vectors = []
    for name, v in [('v_neg_to_pos', v_neg_to_pos), ('v_pos_to_neg', v_pos_to_neg)]:
        v = np.asarray(v, dtype=np.float32)
        assert v.shape == (EMBED_DIM,),  f'{name}: shape {v.shape}, need ({EMBED_DIM},)'
        assert np.isfinite(v).all(),     f'{name}: contains NaN/Inf'
        n = float(np.linalg.norm(v))
        assert n < NORM_CAP,             f'{name}: L2 norm {n:.6f} must be < {NORM_CAP}'
        vectors.append(v)
        print(f'{name} OK  norm={n:.4f}')
    arr = np.stack(vectors, axis=0)   # shape (2, 384)
    np.savetxt(path, arr, delimiter=',', fmt='%.8f')
    print(f'wrote {path}  shape={arr.shape}')

In [248]:
save_submission(v_neg_to_pos, v_pos_to_neg, path=r'C:\Users\asus\LeagueIV_P2\submission.csv')

v_neg_to_pos OK  norm=0.9990
v_pos_to_neg OK  norm=0.9990
wrote C:\Users\asus\LeagueIV_P2\submission.csv  shape=(2, 384)


In [268]:
df_submission = pd.DataFrame([v_neg_to_pos, v_pos_to_neg])

# 2. This saves it without any headers or row numbers
df_submission.to_csv("contestant_output.csv", index=False, header=False)

## 7. Your work

Zero and random barely flip anything. A well-chosen direction with norm just under 1.0 does.

Hint: `get_layer3_activations(texts)` gives you `(N, 384)` activations at the exact point where your steering vector will be added. What happens if you feed it sentiment-laden phrases?
